# 10 | Campaign-ready windows: the 2026 ATP calendar scored for India

**Author: Chanakya**

Every 2026 ATP tour event in FanCode’s package, split into three viewing windows (day session, night session, final) and scored for an Indian audience. A window is **campaign-ready** only if its start sits inside every one of the three viewing windows under every start delay from 0 to 120 minutes (12 of 12 tests), the same standard applied to the verified finals in notebook 01. Each window carries its clash check, player-story trigger, offer and timing confidence. The logic lives in `models/campaign_windows.py`. Player draws are not scored: they are unknown until the week of the event.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='10_campaign_ready_windows'
shared.ACTIVE_SOURCES=[]

Offline inputs: raw-v5-2026-09-14 | Author: Chanakya


In [2]:
sys.path.insert(0, str(ROOT/'models'))
import campaign_windows as cw
windows, events = cw.build()
summary = cw.summarise(windows, events)
display(table(windows,'10_campaign_ready_windows'))
(ROOT/'outputs/reports/10_campaign_windows_summary.json').write_text(json.dumps({'settings':cw.SETTINGS,'summary':summary},indent=1,default=str)+'\n')
settings=pd.DataFrame([dict(setting=k,value=json.dumps(v) if not isinstance(v,str) else v) for k,v in cw.SETTINGS.items()]);display(table(settings,'10_window_scoring_settings'))

,rank,event,tier,event_start,final_date,final_date_basis,session,local_start,local_timezone,...,timing_cells_passed,status,clash_check,continuity_trigger,offer,rights,timing_confidence,upcoming,score
0,1,Paris,Masters 1000,2026-11-02,2026-11-08,ATP calendar end date (O),Final,15:00 (modelled),Europe/Paris,...,12/12,Campaign-ready: sell live,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 99 tournament pass at list,Covered: ATP Tour package (C),"Medium: published date, modelled time (A)",True,92.500
1,2,Turin,Finals,2026-11-15,2026-11-22,ATP calendar end date (O),Final,15:00 (modelled),Europe/Rome,...,12/12,Campaign-ready: sell live,Football not checked (no 2026-27 fixture data ...,Season finale,INR 99 tournament pass at list,Covered: ATP Tour package (C),"Medium: published date, modelled time (A)",True,92.500
2,3,Paris,Masters 1000,2026-11-02,2026-11-08,ATP calendar end date (O),Day session,14:00 (modelled),Europe/Paris,...,11/12,Early evening: live with start reminder,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 99 tournament pass at list; reminder at st...,Covered: ATP Tour package (C),"Medium: published date, modelled time (A)",True,89.200
3,4,Turin,Finals,2026-11-15,2026-11-22,ATP calendar end date (O),Day session,14:00 (modelled),Europe/Rome,...,11/12,Early evening: live with start reminder,Football not checked (no 2026-27 fixture data ...,Season finale,INR 99 tournament pass at list; reminder at st...,Covered: ATP Tour package (C),"Medium: published date, modelled time (A)",True,89.200
4,5,Halle,500,2026-06-15,2026-06-21,Official order of play (O),Final,15:30 (not before),Europe/Berlin,...,12/12,Campaign-ready: sell live,No overlap found,Post-Slam follow-through: Roland Garros,INR 89 tournament pass at list,Covered: ATP Tour package (C),High: official order of play (O),False,88.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,161,Delray Beach,250,2026-02-16,2026-02-22,Weekly event rule (A),Final,15:00 (modelled),America/New_York,...,0/12,Overnight: replay only,La Liga fixtures overlapping: 1,No trigger,INR 39 replay where rights permit,Covered: ATP Tour package (C),"Low: inherited date, modelled time (A)",False,16.500
161,162,Santiago,250,2026-02-23,2026-03-01,Weekly event rule (A),Night session,19:00 (modelled),America/Santiago,...,0/12,Overnight: replay only,La Liga fixtures overlapping: 3,No trigger,INR 39 replay where rights permit,Covered: ATP Tour package (C),"Low: inherited date, modelled time (A)",False,16.500
162,163,Houston,250,2026-03-30,2026-04-05,Weekly event rule (A),Day session,14:00 (modelled),America/Chicago,...,0/12,Overnight: replay only,La Liga fixtures overlapping: 2,No trigger,INR 39 replay where rights permit,Covered: ATP Tour package (C),"Low: inherited date, modelled time (A)",False,16.500
163,164,Houston,250,2026-03-30,2026-04-05,Weekly event rule (A),Final,15:00 (modelled),America/Chicago,...,0/12,Overnight: replay only,La Liga fixtures overlapping: 1,No trigger,INR 39 replay where rights permit,Covered: ATP Tour package (C),"Low: inherited date, modelled time (A)",False,16.500


,setting,value
0,_about,"Scoring settings. Weights are choices, not est..."
1,session_local_start,"{""Day session"": ""14:00"", ""Night session"": ""19:..."
2,viewing_windows_ist,"[[18, 23], [19, 24], [17, 24]]"
3,delays_minutes,"[0, 30, 60, 120]"
4,weights,"{""timing"": 0.4, ""tier"": 0.3, ""continuity"": 0.1..."
5,tier_value,"{""ATP FINALS"": 1.0, ""ATP MASTERS 1000"": 1.0, ""..."
6,list_price,"{""ATP FINALS"": 99, ""ATP MASTERS 1000"": 99, ""AT..."
7,replay_price,39
8,post_slam_days,10
9,race_to_turin_days,42


## 1. How the season divides

In [3]:
counts=windows.groupby(['session','status']).size().unstack(fill_value=0);display(counts)
st=windows.status.value_counts().rename_axis('status').reset_index(name='windows');display(table(st,'10_window_status_counts'))
print(f"{summary['campaign_ready_live']} of {summary['windows']} windows across {summary['events']} events are campaign-ready; {summary['upcoming_live_windows']} are still ahead after 18 September 2026.")

status,Campaign-ready: sell live,Daytime: highlights,Early evening: live with start reminder,Late: remind + replay,Overnight: replay only
session,,,,,
Day session,0,9,31,8,7
Final,17,9,14,5,10
Night session,2,5,6,30,12


,status,windows
0,Early evening: live with start reminder,51
1,Late: remind + replay,43
2,Overnight: replay only,29
3,Daytime: highlights,23
4,Campaign-ready: sell live,19


19 of 165 windows across 55 events are campaign-ready; 8 are still ahead after 18 September 2026.


In [4]:
colors={'Campaign-ready: sell live':COLORS[1],'Early evening: live with start reminder':COLORS[0],'Late: remind + replay':COLORS[2],'Overnight: replay only':COLORS[3],'Daytime: highlights':COLORS[5]}
size={'Finals':140,'Masters 1000':110,'500':70,'250':40}
fig_,ax=plt.subplots(figsize=(12,4.8))
w=windows.copy();w['d']=pd.to_datetime(w.window_date);w['h']=[int(x[:2])+int(x[3:])/60 for x in w.ist_start]
for s_,g in w.groupby('status'):ax.scatter(g.d,g.h,s=[size[t] for t in g.tier],color=colors[s_],alpha=.8,label=s_,edgecolor='white',linewidth=.5)
ax.axhspan(18,23,color=COLORS[1],alpha=.08);ax.set_ylim(0,24);ax.set_yticks(range(0,25,3));ax.set_ylabel('Start, IST hour');ax.legend(fontsize=8,loc='upper center',bbox_to_anchor=(.5,-.08),ncol=5,frameon=False)
ax.axvline(pd.Timestamp('2026-09-18'),color='black',ls=':');ax.text(pd.Timestamp('2026-09-20'),1,'today',fontsize=8)
ax.set_title('The Europe and Gulf swing lands in Indian prime time; the Americas and Asia do not')
fig('10_campaign_windows_calendar','Marker size = tier. Shaded band = 18:00-23:00 IST. Session times are modelled local conventions except the 12 finals verified from official orders of play.')

<Figure size 1200x480 with 1 Axes>

## 2. The list: campaign-ready windows, ranked
Score out of 100 = 40% timing robustness + 30% tier + 15% player-story continuity (post-Slam follow-through, race to Turin, season finale) + 15% clash-free. Weights are settings, not estimates. A clash does not disqualify a window: it changes who receives the send.

In [5]:
ready=windows[windows.status=='Campaign-ready: sell live'][['rank','event','tier','session','window_date','ist_start','clash_check','continuity_trigger','offer','timing_confidence','upcoming','score']]
display(table(ready,'10_campaign_ready_list'))
ahead=ready[ready.upcoming];display(table(ahead,'10_campaign_ready_upcoming'))

,rank,event,tier,session,window_date,ist_start,clash_check,continuity_trigger,offer,timing_confidence,upcoming,score
0,1,Paris,Masters 1000,Final,2026-11-08,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 99 tournament pass at list,"Medium: published date, modelled time (A)",True,92.500
1,2,Turin,Finals,Final,2026-11-22,19:30,Football not checked (no 2026-27 fixture data ...,Season finale,INR 99 tournament pass at list,"Medium: published date, modelled time (A)",True,92.500
4,5,Halle,500,Final,2026-06-21,19:00,No overlap found,Post-Slam follow-through: Roland Garros,INR 89 tournament pass at list,High: official order of play (O),False,88.000
5,6,London (Queen's),500,Final,2026-06-21,19:30,No overlap found,Post-Slam follow-through: Roland Garros,INR 89 tournament pass at list,"Low: inherited date, modelled time (A)",False,88.000
7,8,Rotterdam,500,Final,2026-02-15,20:00,La Liga fixtures overlapping: 2,Post-Slam follow-through: Australian Open,INR 89 tournament pass at list,High: official order of play (O),False,80.500
8,9,Basel,500,Final,2026-11-01,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 89 tournament pass at list,"Medium: published date, modelled time (A)",True,80.500
9,10,Vienna,500,Final,2026-11-01,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 89 tournament pass at list,"Low: inherited date, modelled time (A)",True,80.500
10,11,Madrid,Masters 1000,Final,2026-05-03,20:30,La Liga fixtures overlapping: 2,No trigger,INR 99 tournament pass at list,High: official order of play (O),False,77.500
11,12,Rome,Masters 1000,Final,2026-05-17,20:30,La Liga fixtures overlapping: 9,No trigger,INR 99 tournament pass at list,High: official order of play (O),False,77.500
19,20,Montpellier,250,Final,2026-02-08,19:30,La Liga fixtures overlapping: 3,Post-Slam follow-through: Australian Open,INR 79 tournament pass at list,"Medium: published date, modelled time (A)",False,71.500


,rank,event,tier,session,window_date,ist_start,clash_check,continuity_trigger,offer,timing_confidence,upcoming,score
0,1,Paris,Masters 1000,Final,2026-11-08,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 99 tournament pass at list,"Medium: published date, modelled time (A)",True,92.500
1,2,Turin,Finals,Final,2026-11-22,19:30,Football not checked (no 2026-27 fixture data ...,Season finale,INR 99 tournament pass at list,"Medium: published date, modelled time (A)",True,92.500
8,9,Basel,500,Final,2026-11-01,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 89 tournament pass at list,"Medium: published date, modelled time (A)",True,80.500
9,10,Vienna,500,Final,2026-11-01,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 89 tournament pass at list,"Low: inherited date, modelled time (A)",True,80.500
20,21,Almaty,250,Night session,2026-10-19,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 79 tournament pass at list,"Medium: published date, modelled time (A)",True,71.500
21,22,Brussels,250,Final,2026-10-25,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 79 tournament pass at list,"Low: inherited date, modelled time (A)",True,71.500
22,23,Lyon,250,Final,2026-10-25,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 79 tournament pass at list,"Low: inherited date, modelled time (A)",True,71.500
23,24,Stockholm,250,Final,2026-11-14,19:30,Football not checked (no 2026-27 fixture data ...,Race to Turin,INR 79 tournament pass at list,"Medium: published date, modelled time (A)",True,71.500


## 3. Checks

In [6]:
checks={'fifty_five_dated_events':summary['events']==55,
 'three_windows_per_event':len(windows)==3*summary['events'],
 'status_counts_reconcile':sum(v for k,v in summary.items() if k in ['campaign_ready_live','early_evening_live','late_remind_replay','daytime_highlights','overnight_replay'])==len(windows),
 'verified_finals_used':summary['high_confidence_windows']==12,
 'rotterdam_final_2000_ist':windows[(windows.event=='Rotterdam')&(windows.session=='Final')].ist_start.iloc[0]=='20:00',
 'indian_wells_final_overnight':windows[(windows.event=='Indian Wells')&(windows.session=='Final')].status.iloc[0]=='Overnight: replay only',
 'six_verified_finals_robust':int(((windows.session=='Final')&windows.timing_confidence.str.startswith('High')&(windows.status=='Campaign-ready: sell live')).sum())==6,
 'unique_ranks':windows['rank'].is_unique}
check('10_campaign_windows',checks)
top=', '.join(f"{r.event} {r.session.lower()} {r.window_date} {r.ist_start} IST" for r in ahead.head(8).itertuples())
report('10_campaign_windows_findings',f"Of {summary['windows']} windows across {summary['events']} 2026 ATP events in FanCode's package, {summary['campaign_ready_live']} are campaign-ready: their start holds inside Indian prime time under every timing test. {summary['early_evening_live']} more start in the early evening and can be sold live with a start reminder; {summary['late_remind_replay']} are late and get reminders plus the INR 39 replay; {summary['overnight_replay']} are overnight and {summary['daytime_highlights']} daytime. {summary['upcoming_live_windows']} campaign-ready windows are still ahead this season: {top}. Timing confidence is high only for the 12 finals verified from official orders of play; the rest use modelled local session times and must be confirmed from each week's order of play before a send. Football clashes after 8 September 2026 are not yet checked because 2026-27 fixtures were not captured.")

,check,passed
0,fifty_five_dated_events,True
1,three_windows_per_event,True
2,status_counts_reconcile,True
3,verified_finals_used,True
4,rotterdam_final_2000_ist,True
5,indian_wells_final_overnight,True
6,six_verified_finals_robust,True
7,unique_ranks,True
